# Interactive AWGN Communication System
**CS 515 Deep Learning — Homework 4**

Transformer-based encoder/decoder with T=4 feedback rounds, σ²=0.25.

Run all cells top-to-bottom. Training takes ~5-10 min on a T4 GPU.

## 0 · Check GPU

In [ ]:
!nvidia-smi
import torch; print('CUDA:', torch.cuda.is_available())

## 1 · Write module files
> Each `%%writefile` cell saves a `.py` file to Colab's filesystem.

In [ ]:
%%writefile config.py
from dataclasses import dataclass
import torch


@dataclass
class Config:
    # ── Message ──────────────────────────────────────────────
    vocab_size: int = 8          # alphabet {1,...,8}, we use 0-indexed internally
    seq_len:    int = 4          # 4 symbols per message

    # ── Channel ──────────────────────────────────────────────
    T:      int   = 4            # communication rounds
    sigma2: float = 0.25         # AWGN noise variance  →  σ = 0.5

    # ── Model ────────────────────────────────────────────────
    d_model:  int   = 64
    n_heads:  int   = 4
    n_layers: int   = 2
    d_ff:     int   = 128
    d_emb:    int   = 32         # symbol embedding dim (fed into pre-MLP)
    dropout:  float = 0.1

    # ── Training ─────────────────────────────────────────────
    batch_size: int   = 256
    lr:         float = 3e-4
    epochs:     int   = 150
    train_size: int   = 64_000
    val_size:   int   = 8_000
    grad_clip:  float = 1.0

    # ── Misc ─────────────────────────────────────────────────
    seed:       int  = 42
    device:     str  = "cuda"
    save_path:  str  = "model.pt"

    def get_device(self) -> torch.device:
        return torch.device(self.device if torch.cuda.is_available() else "cpu")


In [ ]:
%%writefile data.py
import torch
from torch.utils.data import Dataset, DataLoader
from config import Config


class MessageDataset(Dataset):
    """
    Each sample is a message m ∈ {0,...,7}^4  (0-indexed internally;
    the paper uses {1,...,8} but shifting by 1 has no effect on training).
    """
    def __init__(self, size: int, config: Config):
        torch.manual_seed(config.seed if size == config.train_size else config.seed + 1)
        self.data = torch.randint(0, config.vocab_size, (size, config.seq_len))

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> torch.Tensor:
        return self.data[idx]   # (4,)  int64


def get_dataloaders(config: Config):
    train_ds = MessageDataset(config.train_size, config)
    val_ds   = MessageDataset(config.val_size,   config)

    train_dl = DataLoader(
        train_ds, batch_size=config.batch_size,
        shuffle=True,  num_workers=2, pin_memory=True
    )
    val_dl = DataLoader(
        val_ds, batch_size=config.batch_size,
        shuffle=False, num_workers=2, pin_memory=True
    )
    return train_dl, val_dl


In [ ]:
import os
os.makedirs('models', exist_ok=True)

In [ ]:
%%writefile models/__init__.py
from .encoder import TXEncoder
from .decoder import RXDecoder
from .transformer import TransformerStack, TransformerBlock


In [ ]:
%%writefile models/transformer.py
import math
import torch
import torch.nn as nn


# ─────────────────────────────────────────────────────────────────────────────
# Multi-head self-attention
# ─────────────────────────────────────────────────────────────────────────────
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_k     = d_model // n_heads
        self.n_heads = n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, D)
        B, L, D = x.shape
        Q = self.W_q(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)  # (B,H,L,dk)
        K = self.W_k(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)             # (B,H,L,L)
        attn   = self.dropout(torch.softmax(scores, dim=-1))
        out    = (attn @ V).transpose(1, 2).contiguous().view(B, L, D)       # (B,L,D)
        return self.W_o(out)


# ─────────────────────────────────────────────────────────────────────────────
# Position-wise feed-forward network
# ─────────────────────────────────────────────────────────────────────────────
class FFN(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ─────────────────────────────────────────────────────────────────────────────
# Single transformer block — post-norm as given in the problem:
#   H^(l) = LayerNorm( H^(l-1) + MultiHead(H^(l-1)) )
#   H^(l) = LayerNorm( H^(l)   + FFN(H^(l))          )
# ─────────────────────────────────────────────────────────────────────────────
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attn  = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ffn   = FFN(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.norm1(x + self.attn(x))   # Eq. (2)
        x = self.norm2(x + self.ffn(x))    # Eq. (3)
        return x


# ─────────────────────────────────────────────────────────────────────────────
# Stack of transformer blocks with learned positional encoding
# ─────────────────────────────────────────────────────────────────────────────
class TransformerStack(nn.Module):
    """
    H^(0) = Z + PE                      (Eq. 1)
    H^(l) = TransformerBlock(H^(l-1))   (Eqs. 2-3)
    """
    def __init__(
        self,
        d_model:  int,
        n_heads:  int,
        d_ff:     int,
        n_layers: int,
        seq_len:  int,
        dropout:  float = 0.1,
    ):
        super().__init__()
        # Learned positional encoding — shape (1, seq_len, d_model)
        self.pe = nn.Parameter(torch.zeros(1, seq_len, d_model))
        nn.init.trunc_normal_(self.pe, std=0.02)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, Z: torch.Tensor) -> torch.Tensor:
        # Z: (B, seq_len, d_model)
        x = Z + self.pe          # H^(0)
        for block in self.blocks:
            x = block(x)
        return x                 # (B, seq_len, d_model)


In [ ]:
%%writefile models/encoder.py
import torch
import torch.nn as nn
from .transformer import TransformerStack
from config import Config


class TXEncoder(nn.Module):
    """
    At round t the encoder receives:
      • messages     : (B, 4)        — original symbols (fixed across rounds)
      • tx_history   : (B, 4, T-1)  — coded symbols sent in rounds 1…t-1  (zero-padded)
      • fb_history   : (B, 4, T-1)  — feedback y received in rounds 1…t-1 (zero-padded)

    It produces:
      • x_t          : (B, 4)        — power-normalised coded symbols

    Architecture (Hint 3):
        raw input → pre-MLP → TransformerStack → post-MLP → power normalisation
    """

    def __init__(self, config: Config):
        super().__init__()
        self.config = config

        # ── Symbol embedding  m_i ∈ {0,...,7} → R^{d_emb} ──────────────────
        self.symbol_embed = nn.Embedding(config.vocab_size, config.d_emb)

        # ── Pre-MLP: maps raw token → d_model ───────────────────────────────
        # raw dim per token = d_emb  +  (T-1) tx history  +  (T-1) fb history
        raw_dim = config.d_emb + 2 * (config.T - 1)
        self.pre_mlp = nn.Sequential(
            nn.Linear(raw_dim, config.d_model),
            nn.GELU(),
            nn.Linear(config.d_model, config.d_model),
        )

        # ── Transformer ──────────────────────────────────────────────────────
        self.transformer = TransformerStack(
            d_model  = config.d_model,
            n_heads  = config.n_heads,
            d_ff     = config.d_ff,
            n_layers = config.n_layers,
            seq_len  = config.seq_len,
            dropout  = config.dropout,
        )

        # ── Post-MLP: penultimate latent → 1 coded symbol per position ───────
        self.post_mlp = nn.Sequential(
            nn.Linear(config.d_model, config.d_model // 2),
            nn.GELU(),
            nn.Linear(config.d_model // 2, 1),
        )

    def forward(
        self,
        messages:   torch.Tensor,   # (B, 4)        int64
        tx_history: torch.Tensor,   # (B, 4, T-1)   float
        fb_history: torch.Tensor,   # (B, 4, T-1)   float
    ) -> torch.Tensor:              # (B, 4)         float

        # 1. Symbol embedding  →  (B, 4, d_emb)
        sym = self.symbol_embed(messages)           # (B, 4, d_emb)

        # 2. Concatenate histories  →  (B, 4, raw_dim)
        raw = torch.cat([sym, tx_history, fb_history], dim=-1)

        # 3. Pre-MLP  →  Z(t) ∈ (B, 4, d_model)
        Z = self.pre_mlp(raw)

        # 4. Transformer  →  (B, 4, d_model)
        H = self.transformer(Z)

        # 5. Post-MLP  →  (B, 4, 1)  →  (B, 4)
        x = self.post_mlp(H).squeeze(-1)

        # 6. Per-sample power normalisation so that ||x||² = 1
        #    This guarantees E[||x(t)||²] = 1  ≤  1  ✓
        norm = x.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        x = x / norm

        return x    # (B, 4),  ||x[i]||² = 1 for every sample i


In [ ]:
%%writefile models/decoder.py
import torch
import torch.nn as nn
from .transformer import TransformerStack
from config import Config


class RXDecoder(nn.Module):
    """
    Runs ONCE after all T rounds (Hint 2).

    Input:
      • received : (B, T, 4)  — all noisy signals y(1)…y(T)

    For each symbol position i the decoder sees the vector
      [y_i(1), y_i(2), …, y_i(T)]  ∈  R^T
    giving 4 tokens of dimension T.

    Architecture:
        per-position raw  →  pre-MLP  →  TransformerStack  →  classifier
                                                                  ↓
                                                       logits (B, 4, vocab_size)
    """

    def __init__(self, config: Config):
        super().__init__()
        self.config = config

        # ── Pre-MLP: T received values per position → d_model ───────────────
        self.pre_mlp = nn.Sequential(
            nn.Linear(config.T, config.d_model),
            nn.GELU(),
            nn.Linear(config.d_model, config.d_model),
        )

        # ── Transformer ──────────────────────────────────────────────────────
        self.transformer = TransformerStack(
            d_model  = config.d_model,
            n_heads  = config.n_heads,
            d_ff     = config.d_ff,
            n_layers = config.n_layers,
            seq_len  = config.seq_len,
            dropout  = config.dropout,
        )

        # ── Classification head: d_model → vocab_size ────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(config.d_model, config.d_model // 2),
            nn.GELU(),
            nn.Linear(config.d_model // 2, config.vocab_size),
        )

    def forward(self, received: torch.Tensor) -> torch.Tensor:
        """
        received : (B, T, 4)
        returns  : (B, 4, vocab_size)  — logits for each symbol position
        """
        # Rearrange to (B, 4, T): one token per symbol position
        y = received.permute(0, 2, 1)          # (B, 4, T)

        # Pre-MLP  →  (B, 4, d_model)
        Z = self.pre_mlp(y)

        # Transformer  →  (B, 4, d_model)
        H = self.transformer(Z)

        # Classification  →  (B, 4, vocab_size)
        logits = self.classifier(H)

        return logits


In [ ]:
%%writefile system.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import Config
from models import TXEncoder, RXDecoder


class CommSystem(nn.Module):
    """
    Full interactive communication system for T rounds.

    Forward pass:
      1.  For t = 1 … T:
            a. TX Encoder produces x(t) from (messages, tx_history, fb_history)
            b. AWGN channel:  y(t) = x(t) + ε,  ε ~ N(0, σ²I)
            c. Feedback (Hint 1):  f(t) = y(t)   [noiseless relay]
            d. Update histories for the next round
      2.  RX Decoder maps all {y(t)} → logits (B, 4, 8)
    """

    def __init__(self, config: Config):
        super().__init__()
        self.config  = config
        self.sigma   = config.sigma2 ** 0.5
        self.encoder = TXEncoder(config)
        self.decoder = RXDecoder(config)

    # ─────────────────────────────────────────────────────────────────────────
    def forward(self, messages: torch.Tensor) -> torch.Tensor:
        """
        messages : (B, 4)  int64  ∈ {0,...,7}
        returns  : (B, 4, vocab_size)  logits
        """
        B      = messages.size(0)
        T      = self.config.T
        device = messages.device

        # Running histories — always shape (B, 4, T-1), zero-padded on the right
        # We rebuild them from lists to keep the computation graph intact
        # (avoids in-place ops on tensors that need gradients)
        x_list: list[torch.Tensor] = []   # coded symbols  x(1)…x(t-1)
        y_list: list[torch.Tensor] = []   # feedback        y(1)…y(t-1)
        received_list: list[torch.Tensor] = []

        for t in range(T):

            # ── Build history tensors (B, 4, T-1) ────────────────────────────
            if t == 0:
                tx_hist = torch.zeros(B, self.config.seq_len, T - 1, device=device)
                fb_hist = torch.zeros(B, self.config.seq_len, T - 1, device=device)
            else:
                # Stack what we have so far  →  (B, 4, t)
                tx_hist = torch.stack(x_list, dim=2)
                fb_hist = torch.stack(y_list, dim=2)
                # Pad right to T-1 with zeros
                pad = T - 1 - t
                if pad > 0:
                    tx_hist = F.pad(tx_hist, (0, pad))
                    fb_hist = F.pad(fb_hist, (0, pad))

            # ── TX Encoder  →  x(t) ∈ (B, 4), ||x||²=1 ──────────────────────
            x_t = self.encoder(messages, tx_hist, fb_hist)

            # ── AWGN channel  →  y(t) = x(t) + ε ────────────────────────────
            noise = torch.randn_like(x_t) * self.sigma
            y_t   = x_t + noise

            # ── Store for histories and final decoding ────────────────────────
            x_list.append(x_t)
            y_list.append(y_t)           # feedback = y(t)  (Hint 1)
            received_list.append(y_t)

        # ── Stack all received signals  →  (B, T, 4) ─────────────────────────
        received = torch.stack(received_list, dim=1)   # (B, T, 4)

        # ── RX Decoder (runs once)  →  (B, 4, vocab_size) ────────────────────
        logits = self.decoder(received)

        return logits

    # ─────────────────────────────────────────────────────────────────────────
    @torch.no_grad()
    def predict(self, messages: torch.Tensor) -> torch.Tensor:
        """Returns hard symbol predictions  (B, 4)  int64."""
        logits = self.forward(messages)
        return logits.argmax(dim=-1)

    # ─────────────────────────────────────────────────────────────────────────
    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


In [ ]:
%%writefile train.py
import os
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from config import Config
from data import get_dataloaders
from system import CommSystem


# ─────────────────────────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer, device, config, train: bool):
    model.train(train)
    total_loss   = 0.0
    total_correct = 0
    total_symbols = 0

    with torch.set_grad_enabled(train):
        for messages in loader:
            messages = messages.to(device)          # (B, 4)

            logits = model(messages)                # (B, 4, 8)

            # Cross-entropy over all 4 × B symbol predictions
            loss = F.cross_entropy(
                logits.view(-1, config.vocab_size),
                messages.view(-1),
            )

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                optimizer.step()

            preds = logits.argmax(dim=-1)           # (B, 4)
            total_correct  += (preds == messages).sum().item()
            total_symbols  += messages.numel()
            total_loss     += loss.item()

    avg_loss = total_loss / len(loader)
    ser      = 1.0 - total_correct / total_symbols   # Symbol Error Rate
    return avg_loss, ser


# ─────────────────────────────────────────────────────────────────────────────
def train(config: Config | None = None):
    if config is None:
        config = Config()

    torch.manual_seed(config.seed)
    device = config.get_device()
    print(f"Using device: {device}")

    # ── Data ─────────────────────────────────────────────────────────────────
    train_dl, val_dl = get_dataloaders(config)

    # ── Model ────────────────────────────────────────────────────────────────
    model = CommSystem(config).to(device)
    print(f"Parameters: {model.count_parameters():,}")

    # ── Optimiser + Scheduler ────────────────────────────────────────────────
    optimizer = AdamW(model.parameters(), lr=config.lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=config.epochs, eta_min=1e-5)

    # ── Training loop ────────────────────────────────────────────────────────
    best_val_ser = float("inf")
    history = {"train_loss": [], "val_loss": [], "train_ser": [], "val_ser": []}

    for epoch in range(1, config.epochs + 1):
        tr_loss, tr_ser = run_epoch(model, train_dl, optimizer, device, config, train=True)
        va_loss, va_ser = run_epoch(model, val_dl,   optimizer, device, config, train=False)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_ser"].append(tr_ser)
        history["val_ser"].append(va_ser)

        if epoch % 10 == 0 or epoch == 1:
            lr_now = scheduler.get_last_lr()[0]
            print(
                f"Epoch {epoch:4d}/{config.epochs} | "
                f"Train loss {tr_loss:.4f}  SER {tr_ser:.4f} | "
                f"Val loss {va_loss:.4f}  SER {va_ser:.4f} | "
                f"LR {lr_now:.2e}"
            )

        # Save best checkpoint
        if va_ser < best_val_ser:
            best_val_ser = va_ser
            torch.save(
                {"epoch": epoch, "model_state": model.state_dict(),
                 "val_ser": va_ser, "config": config},
                config.save_path,
            )

    print(f"\nBest val SER: {best_val_ser:.4f}  (checkpoint → {config.save_path})")
    return model, history


# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    train()


In [ ]:
%%writefile evaluate.py
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from config import Config
from data import MessageDataset
from system import CommSystem
from torch.utils.data import DataLoader


# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model: CommSystem, config: Config, n_test: int = 10_000):
    """
    Returns a dict with:
      ser           – overall Symbol Error Rate
      per_pos_ser   – SER per symbol position (length 4)
      confusion     – (8, 8) confusion matrix (row=true, col=pred)
    """
    device = config.get_device()
    model.eval()

    ds  = MessageDataset(n_test, config)
    dl  = DataLoader(ds, batch_size=512, shuffle=False)

    all_true, all_pred = [], []

    for messages in dl:
        messages = messages.to(device)
        preds    = model.predict(messages)
        all_true.append(messages.cpu())
        all_pred.append(preds.cpu())

    true = torch.cat(all_true)   # (N, 4)
    pred = torch.cat(all_pred)   # (N, 4)

    ser         = (true != pred).float().mean().item()
    per_pos_ser = (true != pred).float().mean(dim=0).tolist()   # list of 4

    # Confusion matrix (aggregate over all positions)
    conf = torch.zeros(config.vocab_size, config.vocab_size, dtype=torch.long)
    for t, p in zip(true.view(-1).tolist(), pred.view(-1).tolist()):
        conf[t][p] += 1

    return {"ser": ser, "per_pos_ser": per_pos_ser, "confusion": conf.numpy()}


# ─────────────────────────────────────────────────────────────────────────────
def plot_results(history: dict, eval_results: dict, save_path: str = "results.png"):
    fig = plt.figure(figsize=(16, 10))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    epochs = range(1, len(history["train_loss"]) + 1)

    # 1 — Training & validation loss
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(epochs, history["train_loss"], label="Train")
    ax1.plot(epochs, history["val_loss"],   label="Val")
    ax1.set_title("Cross-entropy Loss")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    # 2 — Symbol Error Rate curves
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.semilogy(epochs, history["train_ser"], label="Train SER")
    ax2.semilogy(epochs, history["val_ser"],   label="Val SER")
    ax2.set_title("Symbol Error Rate (log scale)")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("SER")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    # 3 — Per-position SER bar chart
    ax3 = fig.add_subplot(gs[0, 2])
    pos_labels = [f"Pos {i+1}" for i in range(4)]
    ax3.bar(pos_labels, eval_results["per_pos_ser"], color="steelblue")
    ax3.set_title("SER per Symbol Position")
    ax3.set_ylabel("SER"); ax3.set_ylim(0, max(eval_results["per_pos_ser"]) * 1.3 + 1e-6)
    ax3.grid(True, axis="y", alpha=0.3)
    for i, v in enumerate(eval_results["per_pos_ser"]):
        ax3.text(i, v + 0.001, f"{v:.4f}", ha="center", fontsize=9)

    # 4 — Confusion matrix
    ax4 = fig.add_subplot(gs[1, :2])
    conf = eval_results["confusion"].astype(float)
    conf_norm = conf / conf.sum(axis=1, keepdims=True).clip(min=1)
    im = ax4.imshow(conf_norm, cmap="Blues", vmin=0, vmax=1)
    ax4.set_title("Normalised Confusion Matrix")
    ax4.set_xlabel("Predicted symbol")
    ax4.set_ylabel("True symbol")
    ticks = list(range(8))
    ax4.set_xticks(ticks); ax4.set_xticklabels([str(i+1) for i in ticks])
    ax4.set_yticks(ticks); ax4.set_yticklabels([str(i+1) for i in ticks])
    plt.colorbar(im, ax=ax4, fraction=0.046, pad=0.04)

    # 5 — Summary text box
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.axis("off")
    summary = (
        f"Final Results\n"
        f"─────────────────\n"
        f"Overall SER : {eval_results['ser']:.4f}\n\n"
        f"Per-position SER\n"
        + "\n".join(
            f"  Position {i+1} : {v:.4f}"
            for i, v in enumerate(eval_results["per_pos_ser"])
        )
        + f"\n\nBest val SER  : {min(history['val_ser']):.4f}"
        + f"\n(epoch {np.argmin(history['val_ser'])+1})"
    )
    ax5.text(
        0.1, 0.95, summary,
        transform=ax5.transAxes, fontsize=11,
        verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8),
    )

    plt.suptitle(
        f"Interactive AWGN Communication  |  T={4}, σ²=0.25",
        fontsize=13, fontweight="bold"
    )
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figure saved → {save_path}")


# ─────────────────────────────────────────────────────────────────────────────
def load_model(checkpoint_path: str, config: Config) -> CommSystem:
    device = config.get_device()
    ckpt   = torch.load(checkpoint_path, map_location=device)
    model  = CommSystem(config).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    print(f"Loaded checkpoint from epoch {ckpt['epoch']}  (val SER={ckpt['val_ser']:.4f})")
    return model


# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    config  = Config()
    model   = load_model(config.save_path, config)
    results = evaluate(model, config)
    print(f"Test SER: {results['ser']:.4f}")
    print(f"Per-pos : {[f'{s:.4f}' for s in results['per_pos_ser']]}")


## 2 · Quick sanity check
Verify shapes with a single forward pass before committing to full training.

In [ ]:
import sys, importlib
sys.path.insert(0, '.')

import torch
from config import Config
from system import CommSystem

cfg    = Config()
device = cfg.get_device()
model  = CommSystem(cfg).to(device)
print(f'Parameters: {model.count_parameters():,}')

# Fake batch
msgs   = torch.randint(0, 8, (4, 4)).to(device)
logits = model(msgs)
print(f'Input  shape : {msgs.shape}     (B=4, seq=4)')
print(f'Output shape : {logits.shape}   (B=4, seq=4, vocab=8)')
print('Sanity check passed ✓')


## 3 · Train
Full training run (~150 epochs). Progress printed every 10 epochs.

In [ ]:
from train import train
from config import Config

cfg = Config()
model, history = train(cfg)


## 4 · Evaluate & plot results

In [ ]:
from evaluate import evaluate, plot_results, load_model
from config import Config

cfg     = Config()
model   = load_model(cfg.save_path, cfg)
results = evaluate(model, cfg, n_test=10_000)

print(f'\nTest SER  : {results["ser"]:.4f}')
print(f'Per-pos   : {[f"{s:.4f}" for s in results["per_pos_ser"]]}')

plot_results(history, results, save_path='results.png')


## 5 · (Optional) Experiment with hyperparameters
Try changing `d_model`, `n_layers`, `T`, or `sigma2` in `Config`.

In [ ]:
# Example: deeper model
from config import Config
from train import train

cfg_deep = Config(
    d_model  = 128,
    n_layers = 4,
    d_ff     = 256,
    epochs   = 100,
    save_path= 'model_deep.pt',
)
model_deep, hist_deep = train(cfg_deep)
